In [1]:
# ignore the unnecessary warning messages
import warnings
warnings.filterwarnings("ignore")

In [2]:
# if document is too long, it need to go-through chunking process

# example chunked document
# using multiple user question

import json

with open("document_chunks_and_questions.json", "r", encoding="utf-8") as file:
    data = json.load(file)

examples_chunks_list = data['examples_chunks_list']
user_questions = data['user_questions']

In [3]:
# creating the client - make the connections with ollama api endpoints
import ollama

ollama_client = ollama.Client(
    host="http://127.0.0.1:11434"
)

print("Ollama Client connection is successful")

Ollama Client connection is successful


In [4]:
def embedding_model(content):
    return ollama_client.embed(
        model="nomic-embed-text:latest",
        input=content
    )

chunk_embedding = embedding_model(examples_chunks_list)['embeddings']
questions_embedding = embedding_model(user_questions)['embeddings']

In [ ]:
import yaml

with open("prompts.yaml", "r") as file:
    prompts = yaml.safe_load(file)

system_prompt = prompts["02_system_prompt"]

In [6]:
def call_model(prompt):
    return ollama_client.generate(
        model="phi3:latest",
        prompt=prompt,
        system=system_prompt,
        options={
            "temperature": 0.2,
            "top_p": 0.9,
            "top_k": 40
        },
        stream=False
    )

In [ ]:
# Function to calculate cosine similarity scores between user-question embeddings and document embeddings
# top scored content passed as context for model response with user-question

from sklearn.metrics.pairwise import cosine_similarity

for question_embedding, user_question in zip(questions_embedding, user_questions):
    scores = cosine_similarity(
        [question_embedding],
        chunk_embedding
    )[0]

    print(f"User-Question: {user_question}")

    current = 0
    context = None
    for doc, score in zip(examples_chunks_list, scores):
        if score > current:
            current = score
            context = doc
        print(f"{float(score) * 100:.2f}% - {doc.split('.')[0]}")

    print("\n")

    prompt = prompts["02_user_prompt"].format(
        context=context,
        question=user_question
    )

    response = call_model(prompt)

    print(f"Selected: {float(current) * 100:.2f}% - {context.split('.')[0]}")
    print(f"Response: {response['response']}\n")


User-Question: What is Python and what is it used for?
85.62% - Python is a high-level programming language
46.75% -  RAG stands for Retrieval-Augmented Generation
42.42% -  Vector databases store numerical representations called embeddings
53.69% - Machine learning is a field of artificial intelligence


Selected: 85.62% - Python is a high-level programming language
Response: Python is a high-level programming language. It is commonly used for web development, data science, machine learning, automation, and scripting.

User-Question: How does Retrieval-Augmented Generation (RAG) work?
44.73% - Python is a high-level programming language
80.39% -  RAG stands for Retrieval-Augmented Generation
53.23% -  Vector databases store numerical representations called embeddings
47.82% - Machine learning is a field of artificial intelligence


Selected: 80.39% -  RAG stands for Retrieval-Augmented Generation
Response: Retrieval-Augmented Generation (RAG) works by enabling an LLM to access and ret

# LLM Sampling Parameters: Temperature, Top-K, Top-P, and Candidate Pool

## 1. What is a Candidate Pool?

A **candidate pool** is a collection of possible next tokens (words or parts of words) that an LLM can choose from when generating its response.

When an LLM receives a prompt, it predicts multiple possible next tokens and assigns a probability to each token.

### Example

Prompt:

```text
Python is a ...
```

The model predicts the following possibilities:

| Candidate Token | Probability |
|---|---:|
| programming | 50% |
| language | 25% |
| popular | 15% |
| computer | 7% |
| database | 3% |

All five possible tokens form the **candidate pool**.

The model selects the next token from this pool based on its sampling strategy.

---

## 2. What is Top-K Sampling?

**Top-K** limits the number of candidate tokens the model can consider when generating the next token.

For example:

```python
top_k = 3
```

The model considers only the three most probable candidate tokens.

| Candidate Token | Probability |
|---|---:|
| programming | 50% |
| language | 25% |
| popular | 15% |
| computer | 7% |
| database | 3% |

After applying Top-K = 3:

- programming (50%)
- language (25%)
- popular (15%)

The remaining candidates are excluded.

**Key Point:** Top-K controls the maximum number of candidate tokens considered at each generation step.

---

## 3. What is Top-P Sampling?

**Top-P**, also called Nucleus Sampling, selects candidate tokens based on their cumulative probability.

Unlike Top-K, it does not use a fixed number of candidate tokens.

For example:

```python
top_p = 0.8
```

The model selects the smallest group of highest-probability tokens whose cumulative probability reaches at least 80%.

| Candidate Token | Probability | Cumulative Probability |
|---|---:|---:|
| programming | 50% | 50% |
| language | 25% | 75% |
| popular | 15% | 90% |
| computer | 7% | 97% |
| database | 3% | 100% |

After applying Top-P = 0.8:

- programming (50%)
- language (25%)
- popular (15%)

Total cumulative probability = 90%.

The model includes the third token because the cumulative probability reaches or exceeds 80%.

**Key Point:** Top-P dynamically controls the candidate pool based on cumulative probability.

---

## 4. Difference Between Top-K and Top-P

| Feature | Top-K | Top-P |
|---|---|---|
| Full Form | Top-K Sampling | Nucleus Sampling |
| Controls | Number of candidate tokens | Cumulative probability |
| Selection | Fixed maximum number | Dynamic number of tokens |
| Example | `top_k=40` | `top_p=0.9` |
| Main Purpose | Limits candidate choices | Filters candidates based on probability |

### Easy Way to Remember

- **Top-K:** How many candidate tokens can the model consider?
- **Top-P:** How much cumulative probability should the candidate tokens cover?

---

## 5. What is Temperature?

Temperature controls the randomness of token selection.

It influences how strongly the model favors high-probability tokens over lower-probability tokens.

| Temperature | Behaviour |
|---|---|
| 0.0 | Greedy or nearly deterministic selection |
| 0.2 | More focused and predictable responses |
| 0.5 | Balanced responses |
| 0.8 | More varied and creative responses |
| 1.0 | More diverse token selection |

### Example

For a RAG application:

- Lower temperature helps generate focused, consistent answers.
- Higher temperature allows more variation in generated responses.

**Key Point:** Temperature controls the randomness of token selection, not the length of the response.

---

## 6. How These Parameters Work Together

When an LLM generates a response:

1. The model receives a prompt.
2. It predicts possible next tokens.
3. Each candidate token receives a probability.
4. Top-K limits the number of candidate tokens.
5. Top-P filters candidates based on cumulative probability.
6. Temperature adjusts the probability distribution.
7. The model selects the next token.
8. The process repeats until the response is complete.

---

## 7. Recommended Settings And Final Summary

| Parameter      | Value | Simple Explanation                                | Purpose                         |
| -------------- | ----- | ------------------------------------------------- | ------------------------------- |
| Candidate Pool | —     | Collection of possible next tokens                | Provides possible token choices |
| Temperature    | 0.2   | Controls randomness in token selection            | Encourages focused answers      |
| Top-K          | 40    | Limits the number of candidate tokens             | Limits candidate tokens         |
| Top-P          | 0.9   | Limits candidates based on cumulative probability | Controls cumulative probability |

**Important Note for RAG:**

These parameters control how the LLM generates its response. They do not control document retrieval, embedding generation, cosine similarity, or document ranking.

For RAG answer quality, the relevance of retrieved context and the quality of the system prompt are especially important.